# Version Mismatch Check — Integration Test

This notebook tests the version mismatch detection and migration flow:

1. Install an **old version** (`0.1.109`), create DO+DS state with P2P folders and a job submission
2. Install the **current branch** (dev version)
3. Test **DO login** — should detect mismatch, offer delete
4. Test **DS on old version after DO upgrade** — what does the DS experience?
5. Test **DS login with upgrade** — should detect mismatch, offer archive + delete
6. Verify archive exists on GDrive + cleanup

**Prerequisites**: OAuth tokens in `credentials/` directory.

## Step 1: Install old version and create state

In [1]:
!uv pip install syft-client==0.1.109

Using Python 3.14.3 environment at: /Users/swag/Documents/Coding/syft-client/.venv
Resolved 81 packages in 154ms                                        
Uninstalled 6 packages in 4ms
Installed 6 packages in 5ms                                 
 - syft-bg==0.3.6 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-bg)
 + syft-bg==0.3.4
 - syft-client==0.1.112 (from file:///Users/swag/Documents/Coding/syft-client)
 + syft-client==0.1.109
 - syft-dataset==0.1.15 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-datasets)
 + syft-dataset==0.1.13
 - syft-job==0.1.34 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-job)
 + syft-job==0.1.32
 - syft-permissions==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-permissions)
 + syft-permissions==0.1.7
 - syft-perms==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-perms)
 + syft-perms==0.1.7


In [2]:
# Restart kernel after install — run this cell, then continue from the next cell
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
import syft_client
from syft_client.sync.syftbox_manager import SyftboxManager
print(f"Installed version: {syft_client.__version__}")

Installed version: 0.1.109


In [2]:
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

In [3]:
# Create DO and DS clients with old version using for_jupyter (the API available in 0.1.109)
do_client = SyftboxManager.for_jupyter(
    email=EMAIL_DO, has_do_role=True, token_path=token_path_do
)
ds_client = SyftboxManager.for_jupyter(
    email=EMAIL_DS, has_ds_role=True, token_path=token_path_ds
)

# Set up peer relationship
ds_client.add_peer(EMAIL_DO)
do_client.load_peers()
do_client.approve_peer_request(EMAIL_DS)
ds_client.load_peers()  # DS picks up DO's approval + version file

# Submit a simple job from DS to DO so there's data in the P2P folders
ds_client.submit_bash_job(user=EMAIL_DO, script='echo "hello from old version"')
ds_client.sync()  # Push job files to GDrive P2P inbox folder
print(f"Created state with version {syft_client.__version__}")

Created user directory: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org
Ensured directories exist: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job
Created user directory: /Users/swag/SyftBox_som.wom.beach@gmail.com/som.wom.beach@gmail.com
🔄 Adding sameer@openmined.org on google_personal.gdrive_files...
✅ Added sameer@openmined.org to google_personal.gdrive_files
✅ Peer sameer@openmined.org added successfully on 1 transport(s)
• google_personal.gdrive_files
🔄 Adding som.wom.beach@gmail.com on google_personal.gdrive_files...
✅ Added som.wom.beach@gmail.com to google_personal.gdrive_files
Approved peer request from som.wom.beach@gmail.com
Created user directory: /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org
Created state with version 0.1.109


In [4]:
do_client.jobs[0].approve()
do_client.process_approved_jobs()

No checkpoints found, downloading all events...
Uploading rolling state with 3 events...
Job 'Job - ec95d5d4' approved successfully!
 Found 1 job(s) in approved status

 Executing job: Job - ec95d5d4
 Inbox: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/Job - ec95d5d4
[sameer@openmined.org][Job - ec95d5d4] hello from old version
 Job Job - ec95d5d4 completed successfully
 Output written to /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job/review/som.wom.beach@gmail.com/Job - ec95d5d4/stdout.txt

 Processed 1 job(s)


In [5]:
# Write version files to GDrive so the mismatch check has something to find
# In 0.1.109, write_own_version is on peer_manager, not SyftboxManager directly
do_client.peer_manager.write_own_version()
ds_client.peer_manager.write_own_version()
print("Version files written for DO and DS (0.1.109)")

Version files written for DO and DS (0.1.109)


## Step 2: Upgrade to current branch

Install the dev version from the local repo. This will change `SYFT_CLIENT_VERSION` to the current value while GDrive still has `0.1.109`.

In [6]:
!uv pip install -e /Users/swag/Documents/Coding/syft-client

Using Python 3.14.3 environment at: /Users/swag/Documents/Coding/syft-client/.venv
Resolved 90 packages in 66ms                                         
Uninstalled 6 packages in 18ms
Installed 6 packages in 10msfile:///Users/swag/Documents/Cod
 - syft-bg==0.3.4
 + syft-bg==0.3.6 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-bg)
 - syft-client==0.1.109
 + syft-client==0.1.112 (from file:///Users/swag/Documents/Coding/syft-client)
 - syft-dataset==0.1.13
 + syft-dataset==0.1.15 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-datasets)
 - syft-job==0.1.32
 + syft-job==0.1.34 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-job)
 - syft-permissions==0.1.7
 + syft-permissions==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-permissions)
 - syft-perms==0.1.7
 + syft-perms==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-perms)


In [7]:
# Restart kernel after upgrade
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [3]:
import syft_client
print(f"Installed version: {syft_client.__version__}")
from syft_client.version import SYFT_CLIENT_VERSION

Installed version: 0.1.112


## Step 3: Test DO login with version mismatch

When you run the next cell, you should see a version mismatch prompt.
Choose **[1] Delete all state** to test the DO purge flow.

In [5]:
from syft_client.sync.login import login_do
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
token_path_do = CREDENTIALS_DIR / "token_do.json"

# This should detect the mismatch and prompt you.
# Choose [1] to delete all state and start fresh.
do_client = login_do(
    email=EMAIL_DO,
    token_path=token_path_do,
    sync=False,
    load_peers=False,
)

Ensured directories exist: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job

⚠️  Version mismatch detected.
  Installed client:  0.1.112
  Local SyftBox:     0.1.112
  Remote SyftBox:    0.1.109

  [1] Delete all state and start fresh with v0.1.112
  [2] Quit



Choice [1/2]:  1


Upgrading to v0.1.112...
Deleted local directory: /Users/swag/SyftBox_sameer@openmined.org
Deleted local directory: /Users/swag/SyftBox_sameer@openmined.org-events
Deleted 15 remote files/folders in 5.53s (including 2 orphaned)
Done. Continuing login.

No checkpoints found, downloading all events...
✅ Connected peer-to-peer to 1 peers via: 


In [6]:
do_client.peers

PeerList([Peer(email='som.wom.beach@gmail.com', platforms=[GdriveFilesPlatform(name='Gdrive Files', module_path='google_personal.gdrive_files')], state=<PeerState.REQUESTED_BY_PEER: 'requested_by_peer'>, version=VersionInfo(syft_client_version='0.1.109', min_supported_syft_client_version='0.1.93', protocol_version='1.0.0', min_supported_protocol_version='1.0.0', updated_at=datetime.datetime(2026, 4, 8, 18, 9, 41, 213105, tzinfo=TzInfo(0))), public_encryption_bundle=None, use_encryption=False)])

In [7]:
# Verify DO version is now updated
from syft_client.gdrive_utils import read_local_version
from syft_client.version import SYFT_CLIENT_VERSION

do_remote = do_client.read_own_version()
do_local = read_local_version(do_client.syftbox_folder)

print(f"Expected version: {SYFT_CLIENT_VERSION}")
print(f"DO remote version: {do_remote.syft_client_version if do_remote else 'None'}")
print(f"DO local version:  {do_local.syft_client_version if do_local else 'None'}")

assert do_remote and do_remote.syft_client_version == SYFT_CLIENT_VERSION, "Remote version should match installed"
assert do_local and do_local.syft_client_version == SYFT_CLIENT_VERSION, "Local version should match installed"
print("DO invariant verified: local = remote = installed")

Expected version: 0.1.112
DO remote version: 0.1.112
DO local version:  0.1.112
DO invariant verified: local = remote = installed


## Step 4: DS on old version after DO upgrade

DO has upgraded and cleaned state. DS is still on 0.1.109. What does the DS experience?

Expected: DS still thinks DO is an accepted peer. But DO's version file is no longer shared with DS, so version checks may fail.

In [8]:
!uv pip install syft-client==0.1.109

Using Python 3.14.3 environment at: /Users/swag/Documents/Coding/syft-client/.venv
Resolved 81 packages in 11ms                                         
Uninstalled 6 packages in 3ms
Installed 6 packages in 6ms                                 
 - syft-bg==0.3.6 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-bg)
 + syft-bg==0.3.4
 - syft-client==0.1.112 (from file:///Users/swag/Documents/Coding/syft-client)
 + syft-client==0.1.109
 - syft-dataset==0.1.15 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-datasets)
 + syft-dataset==0.1.13
 - syft-job==0.1.34 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-job)
 + syft-job==0.1.32
 - syft-permissions==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-permissions)
 + syft-permissions==0.1.7
 - syft-perms==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-perms)
 + syft-perms==0.1.7


In [9]:
# Restart kernel, then re-run the helper/constants cells at the top before continuing
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
import syft_client
from syft_client.sync.syftbox_manager import SyftboxManager
print(f"DS running version: {syft_client.__version__}")

DS running version: 0.1.109


In [2]:
from pathlib import Path 

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

# Create DS client on old version
ds_client_old = SyftboxManager.for_jupyter(
    email=EMAIL_DS, has_ds_role=True, token_path=token_path_ds
)

In [3]:
# What peers does the DS see?
ds_client_old.load_peers()
print(f"DS peers: {ds_client_old.peer_manager.approved_peers}")
print(f"DS peer requests: {ds_client_old.peer_manager.requested_by_me_peers}")

DS peers: [Peer(email='sameer@openmined.org', platforms=[GdriveFilesPlatform(name='Gdrive Files', module_path='google_personal.gdrive_files')], state=<PeerState.ACCEPTED: 'accepted'>, version=None, public_encryption_bundle=None, use_encryption=False)]
DS peer requests: []


In [4]:
# Can the DS read the DO's version? (DO's version file was deleted and recreated without sharing)
EMAIL_DO = "sameer@openmined.org"

peer_version = ds_client_old.peer_manager.load_peer_version(EMAIL_DO)
print(f"DO version as seen by DS: {peer_version}")

DO version as seen by DS: None


In [5]:
# What happens when DS tries to submit a job?
try:
    ds_client_old.submit_bash_job(user=EMAIL_DO, script='echo "test from old DS"')
    print("Job submitted successfully")
except Exception as e:
    print(f"Job submission failed: {type(e).__name__}: {e}")

Job submission failed: VersionUnknownError: Cannot submit job for peer sameer@openmined.org: version information not available. The peer may be running an older version of syft-client that doesn't support version negotiation.


In [6]:
# What happens when DS tries to sync?
try:
    ds_client_old.sync()
    print("Sync completed")
except Exception as e:
    print(f"Sync failed: {type(e).__name__}: {e}")

/Users/swag/Documents/Coding/syft-client/.venv/lib/python3.14/site-packages/syft_client/sync/version/peer_manager.py:302: UserWarning: All connected peers (1) have incompatible versions. You may not be able to submit jobs or load datasets until versions match.
  warnings.warn(


Sync completed


## Step 5: Test DS login with upgrade

Now upgrade the DS to the new version and test the archive flow.

Choose **[1] Archive P2P data and upgrade** to test the archive flow.

In [7]:
!uv pip install -e /Users/swag/Documents/Coding/syft-client

Using Python 3.14.3 environment at: /Users/swag/Documents/Coding/syft-client/.venv
Resolved 90 packages in 15ms                                         
Uninstalled 6 packages in 20ms
Installed 6 packages in 10msm file:///Users/swag/Documents/C
 - syft-bg==0.3.4
 + syft-bg==0.3.6 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-bg)
 - syft-client==0.1.109
 + syft-client==0.1.112 (from file:///Users/swag/Documents/Coding/syft-client)
 - syft-dataset==0.1.13
 + syft-dataset==0.1.15 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-datasets)
 - syft-job==0.1.32
 + syft-job==0.1.34 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-job)
 - syft-permissions==0.1.7
 + syft-permissions==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-permissions)
 - syft-perms==0.1.7
 + syft-perms==0.1.9 (from file:///Users/swag/Documents/Coding/syft-client/packages/syft-perms)


In [8]:
# Restart kernel, then re-run the helper/constants cells at the top before continuing
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
import syft_client
print(f"Installed version: {syft_client.__version__}")
from syft_client.version import SYFT_CLIENT_VERSION
from syft_client.gdrive_utils import read_local_version

Installed version: 0.1.112


In [2]:
from syft_client.sync.login import login_ds
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

# This should detect the mismatch and prompt you.
# Choose [1] to archive P2P data and upgrade.
ds_client = login_ds(
    email=EMAIL_DS,
    token_path=token_path_ds,
    sync=False,
    load_peers=False,
)


⚠️  Version mismatch detected.
  Installed client:  0.1.112
  Local SyftBox:     0.1.112
  Remote SyftBox:    0.1.109

  [1] Archive P2P data to SyftBox_archive/0.1.109/ and upgrade
  [2] Delete all state and start fresh with v0.1.112
  [3] Quit



Choice [1/2/3]:  1


Archiving P2P data and upgrading to v0.1.112...
Archived 2 P2P folders to SyftBox_archive/0.1.109/
Deleted local directory: /Users/swag/SyftBox_som.wom.beach@gmail.com
Deleted local directory: /Users/swag/SyftBox_som.wom.beach@gmail.com-event-messages
Deleted 5 remote files/folders in 0.52s (including 3 orphaned)
Done. Continuing login.

✅ Connected to 0 peers


In [3]:
ds_client.peers

PeerList([])

In [5]:
# Verify DS version is now updated
ds_remote = ds_client.read_own_version()
ds_local = read_local_version(ds_client.syftbox_folder)

print(f"Expected version: {SYFT_CLIENT_VERSION}")
print(f"DS remote version: {ds_remote.syft_client_version if ds_remote else 'None'}")
print(f"DS local version:  {ds_local.syft_client_version if ds_local else 'None'}")

assert ds_remote and ds_remote.syft_client_version == SYFT_CLIENT_VERSION, "Remote version should match installed"
assert ds_local and ds_local.syft_client_version == SYFT_CLIENT_VERSION, "Local version should match installed"
print("DS invariant verified: local = remote = installed")

Expected version: 0.1.112
DS remote version: 0.1.112
DS local version:  0.1.112
DS invariant verified: local = remote = installed


## Step 6: Verify archive exists on GDrive

Check that `SyftBox_archive/0.1.109/` was created at GDrive root with the P2P folders.

In [6]:
from google.oauth2.credentials import Credentials
from syft_client.sync.connections.drive.gdrive_transport import (
    GDriveConnection, SCOPES, build_drive_service
)
from syft_client.sync.connections.drive.gdrive_retry import execute_with_retries

# Use DS credentials to check the archive on DS's drive
creds = Credentials.from_authorized_user_file(str(token_path_ds), SCOPES)
service = build_drive_service(creds)

# Find SyftBox_archive folder at root
query = "name='SyftBox_archive' and mimeType='application/vnd.google-apps.folder' and 'me' in owners and trashed=false"
results = execute_with_retries(service.files().list(q=query, fields="files(id, name)"))
archive_folders = results.get("files", [])
print(f"SyftBox_archive folders found: {len(archive_folders)}")
assert len(archive_folders) > 0, "SyftBox_archive should exist"

archive_id = archive_folders[0]["id"]

# Find version subfolder
query = f"'{archive_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
results = execute_with_retries(service.files().list(q=query, fields="files(id, name)"))
version_folders = results.get("files", [])
print(f"Version subfolders: {[f['name'] for f in version_folders]}")
assert any(f["name"] == "0.1.109" for f in version_folders), "0.1.109 subfolder should exist"

# Check archived P2P folders inside the version folder
version_id = [f for f in version_folders if f["name"] == "0.1.109"][0]["id"]
query = f"'{version_id}' in parents and trashed=false"
results = execute_with_retries(service.files().list(q=query, fields="files(id, name)"))
archived_items = results.get("files", [])
print(f"Archived P2P folders: {[f['name'] for f in archived_items]}")
assert len(archived_items) > 0, "Should have archived P2P folders"
print("\nArchive verification passed!")

SyftBox_archive folders found: 1
Version subfolders: ['0.1.109']
Archived P2P folders: ['syft_datasite#sameer@openmined.org#outbox#som.wom.beach@gmail.com', 'syft_datasite#sameer@openmined.org#inbox#som.wom.beach@gmail.com']

Archive verification passed!


## Cleanup

Delete test state and archive folder from GDrive.

In [7]:
from syft_client.gdrive_utils import delete_syftbox
from syft_client.sync.connections.drive.gdrive_retry import execute_with_retries

# Delete DO state
delete_syftbox(token_path=token_path_do, email=EMAIL_DO, verbose=True)

# Delete DS state
delete_syftbox(token_path=token_path_ds, email=EMAIL_DS, verbose=True)

# Delete the SyftBox_archive folder from DS's drive
if archive_folders:
    for folder in archive_folders:
        execute_with_retries(service.files().delete(fileId=folder["id"]))
        print(f"Deleted archive folder: {folder['name']}")

print("\nCleanup complete!")

Deleted 4 remote files/folders in 1.87s
Deleted local directory: /Users/swag/SyftBox_sameer@openmined.org
Deleted local directory: /Users/swag/SyftBox_sameer@openmined.org-events
Deleted 4 remote files/folders in 1.13s (including 2 orphaned)
Deleted local directory: /Users/swag/SyftBox_som.wom.beach@gmail.com
Deleted archive folder: SyftBox_archive

Cleanup complete!
